# Pelatihan Model NLP Sentimen & Clustering Keluhan (GamerGear)

Notebook ini melatih model Natural Language Processing (NLP) untuk menganalisis sentimen ulasan laptop gaming (positif, netral, negatif) dan mengelompokkan keluhan utama pengguna (seperti isu suhu panas, daya baterai, atau build quality) menggunakan TruncatedSVD dan K-Means Clustering.

### Alur Kerja:
1. **Sel 1: Muat Data Ulasan & Pelabelan Sampel** (Tabel `reviews` SQLite & `sampel_label.csv`)
2. **Sel 2: Ubah Teks ke TF-IDF & Latih Klasifikasi Sentimen** (`LogisticRegression`)
3. **Sel 3: Clustering Keluhan (K-Means pada Ulasan Negatif)** (`TruncatedSVD` & `KMeans`)
4. **Sel 4: Simpan Hasil ke Database** (Tabel `sentiment_results` & `complaint_clusters`)

### Sel 1: Muat Data Ulasan & Pelabelan Sampel
Ambil data dari tabel `reviews`. Ekspor 800–1.500 baris ke file CSV sementara (`sampel_label.csv`), buka di Excel/VS Code, lalu beri kolom tambahan `label_sentimen` (positif, netral, negatif) secara manual, kemudian muat kembali ke notebook.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/gamergear.db')
df_reviews = pd.read_sql_query("SELECT * FROM reviews", conn)

# Simpan sampel untuk dilabeli manual (jalankan sekali saja, lalu beri tanda #)
# df_reviews.sample(800, random_state=42).to_csv('../data/sampel_label.csv', index=False)
df_reviews.head()

,id,produk_id,sumber,teks_asli,teks_bersih,tanggal,id_sumber
0,1,None,YouTube,saya ambil nitro 16 3050 penggunaan 24/7 josji...,ambil nitro bagus kurang baterai boros sama ki...,None,UgzVGPtQUoewKpldBMB4AaABAg
1,2,None,YouTube,Kebutuhan buat main game pilih mana antara TUF...,butuh buat main game pilih mana tuf gaming loq...,None,UgxPiXwB5DDN07ZM7Qt4AaABAg
2,3,None,YouTube,"Butuh pencerahan, kemarin beli tuf gaming ram ...",butuh cerah kemarin beli tuf gaming ram gb ter...,None,Ugznx_C_jTR-UawVaPx4AaABAg
3,4,None,YouTube,ada tambahan 1 laptop lagi dari hp victus 15 y...,tambah laptop hp victus enak banget,None,UgwW3JtQa1VUpniY72d4AaABAg
4,5,None,YouTube,Dri piilahn itu plisss saran paling oke yang m...,piilahn plisss saran paling oke mana bantu sar...,None,UgwMc-VnVbpMQHBnM8x4AaABAg


### Sel 2: Ubah Teks ke TF-IDF & Latih Klasifikasi Sentimen
Setelah file `sampel_label.csv` selesai dilabeli manual, muat file tersebut dan latih model `LogisticRegression` atau `MultinomialNB`, kemudian ekspor model dan vectorizer ke folder `model/`.

In [2]:
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

# Pastikan folder model tersedia
os.makedirs('../model', exist_ok=True)

df_labeled = pd.read_csv('../data/sampel_label.csv').dropna(subset=['teks_bersih', 'label_sentimen'])

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_vec = tfidf.fit_transform(df_labeled['teks_bersih'])
y_label = df_labeled['label_sentimen']

clf = LogisticRegression(max_iter=1000)
clf.fit(X_vec, y_label)

# Simpan vectorizer dan model sentimen ke folder model/
joblib.dump(tfidf, '../model/tfidf_vectorizer.pkl')
joblib.dump(clf, '../model/sentiment_model.pkl')

print(classification_report(y_label, clf.predict(X_vec)))

              precision    recall  f1-score   support

     negatif       1.00      0.19      0.32       109
      netral       0.86      1.00      0.92       917
     positif       0.98      0.78      0.87       276

    accuracy                           0.88      1302
   macro avg       0.95      0.66      0.71      1302
weighted avg       0.90      0.88      0.86      1302



### Sel 3: Clustering Keluhan (K-Means pada Ulasan Negatif)
Saring ulasan bersentimen negatif, reduksi dimensinya dengan `TruncatedSVD`, lalu kelompokkan dengan `KMeans` untuk menemukan topik keluhan utama seperti overheating atau fan noise.

In [3]:
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

# Prediksi sentimen seluruh ulasan
all_vec = tfidf.transform(df_reviews['teks_bersih'].fillna(''))
df_reviews['pred_sentimen'] = clf.predict(all_vec)

# Ambil khusus ulasan negatif untuk clustering
neg_reviews = df_reviews[df_reviews['pred_sentimen'] == 'negatif']
neg_vec = tfidf.transform(neg_reviews['teks_bersih'])

svd = TruncatedSVD(n_components=50, random_state=42)
neg_reduced = svd.fit_transform(neg_vec)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(neg_reduced)

joblib.dump(svd, '../model/svd_model.pkl')
joblib.dump(kmeans, '../model/kmeans_model.pkl')
print("Semua model NLP berhasil disimpan ke folder model/!")

Semua model NLP berhasil disimpan ke folder model/!


### Sel 4: Simpan Hasil ke Database
Petakan hasil prediksi sentimen per aspek ke tabel `sentiment_results` dan rangkuman jumlah ulasan tiap klaster ke tabel `complaint_clusters` menggunakan `.to_sql(..., if_exists='replace', index=False)`.

In [4]:
# 1. Petakan hasil prediksi sentimen ke tabel sentiment_results
proba = clf.predict_proba(all_vec).max(axis=1)

df_sentiment = pd.DataFrame({
    'review_id': df_reviews['id'],
    'aspek': 'umum',
    'label_sentimen': df_reviews['pred_sentimen'],
    'skor': proba.round(4)
})
df_sentiment.to_sql('sentiment_results', conn, if_exists='replace', index=False)

# 2. Rangkuman jumlah ulasan dan kata kunci tiap klaster ke tabel complaint_clusters
feature_names = tfidf.get_feature_names_out()
cluster_centers = svd.inverse_transform(kmeans.cluster_centers_)
cluster_counts = pd.Series(clusters).value_counts().to_dict()

labels_klaster = {
    0: "Isu Mati Total & Mobo",
    1: "Daya & Pengisian Baterai",
    2: "Kerusakan Fisik & Build Quality",
    3: "Suhu Panas & Kipas (Thermals)"
}

cluster_summary = []
for i, center in enumerate(cluster_centers):
    top_indices = center.argsort()[::-1][:6]
    kata_kunci = ", ".join([feature_names[idx] for idx in top_indices])
    cluster_summary.append({
        'cluster_no': i,
        'label': labels_klaster.get(i, f"Klaster Keluhan {i}"),
        'kata_kunci': kata_kunci,
        'jumlah_ulasan': cluster_counts.get(i, 0)
    })

df_complaint_clusters = pd.DataFrame(cluster_summary)
df_complaint_clusters.to_sql('complaint_clusters', conn, if_exists='replace', index=False)
conn.close()

print(f"Tabel sentiment_results diperbarui ({len(df_sentiment)} baris).")
print(f"Tabel complaint_clusters diperbarui ({len(df_complaint_clusters)} klaster keluhan).")
print("Data sentiment_results dan complaint_clusters berhasil disimpan ke database!")

Tabel sentiment_results diperbarui (1302 baris).
Tabel complaint_clusters diperbarui (4 klaster keluhan).
Data sentiment_results dan complaint_clusters berhasil disimpan ke database!
